# <center><font size=10>**Natural Language Processing with Generative AI: Medical Assistant**</font></center>
---

## <span style="color:#87CEEB;">**Problem Statement**</span>

### Business Context

The healthcare sector faces growing challenges in managing large volumes of medical information while maintaining diagnostic speed and accuracy. Clinicians often experience information overload, making it difficult to access reliable, up‑to‑date knowledge when making critical decisions. Streamlined, centralized systems that provide rapid access to trusted medical resources can significantly enhance diagnostic efficiency, support informed decision‑making, and improve overall patient care.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

- **Understand** the challenge of information overload faced by healthcare professionals when accessing vast medical knowledge.  
- **Apply** Retrieval‑Augmented Generation (RAG) techniques to deliver fast, reliable, and context‑aware medical information.  
- **Analyze** how AI‑driven knowledge retrieval can improve diagnostic accuracy and patient outcomes.  
- **Evaluate** the potential of AI systems to standardize clinical decision‑making and care practices.  
- **Create** a functional RAG prototype using trusted medical manuals to demonstrate feasibility and real‑world effectiveness.

### Data Description

The [**Merck Manuals**](medical_diagnosis_manual.pdf) are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

---
## <span style="color:#87CEEB;">**Installing and Importing Necessary Libraries and Dependencies**</span>

In [ ]:
# Mount Google Drive in Colab if needed
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Installation for GPU llama-cpp-python
# conda install -y -c conda-forge llama-cpp-python
# !pip install -q requests pymupdf huggingface_hub pandas
# !pip install -q --upgrade langchain-core langchain langchain-community faiss-cpu

In [1]:
#Libraries for processing dataframes,text
import pandas as pd
import time
import re
import os 
import io
from pathlib import Path
from contextlib import redirect_stdout, redirect_stderr 
from dotenv import load_dotenv

# Load Hugging Face token from .env file
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
import torch
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import FAISS


# Libraries for evaluation
from difflib import SequenceMatcher

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download, login
from llama_cpp import Llama

# For cleaning the manual text
from copy import deepcopy

import logging
logging.getLogger("transformers").setLevel(logging.ERROR)


---
## <span style="color:#87CEEB;">**Utility Functions**</span>

In [2]:
# Function to get response from the model with timing

def get_response(
    llm,
    prompt: str,
    max_tokens: int = 256,
    temperature: float = 0,
    top_p: float = 0.95,
    top_k: int = 50,
    stop=None
):
    """
    Generate a response with:
        - KV cache reset
        - hard max_tokens limit
        - safe stop sequence
        - timing and length stats

    Returns:
        dict with:
            - "text": generated answer
            - "time": seconds elapsed
            - "chars": character length
            - "tokens": approximate token count (whitespace split)
    """

    if stop is None:
        stop = ["</s>", "\n\n\n"]  # safe default

    llm.reset() #

    start = time.time()

    output = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        stream=False,
        stop=stop
    )

    elapsed = time.time() - start
    text = output["choices"][0]["text"].strip()

    stats = {
        "text": text,
        "time": elapsed,
        "chars": len(text),
        "tokens": len(text.split())
    }

    return stats


In [3]:
# Function to run the baseline experiment with default parameters

def run_baseline(llm, questions):
    """
    Baseline experiment with default parameters.
    Uses get_response() which returns a stats dictionary.
    Output format matches the structure of other experiment runs.
    """

    print("\n========== BASELINE (DEFAULT PARAMETERS) ==========\n")
    results = []

    for idx, q in enumerate(questions, start=1):
        print(f"\n--- BASELINE QUESTION {idx} of {len(questions)} ---\n")
        print(f"Q: {q}\n")

        stats = get_response(llm, q)   # returns dict with text, time, chars, tokens, truncated flag

        # Detect truncation (if your get_response returns a flag)
        truncated = stats.get("truncated", False)
        completion_status = "Completed" if not truncated else "Truncated"

        print("ANSWER:\n")
        print(stats["text"])
        print(f"\nTime taken: {stats['time']:.2f} seconds")
        print(f"Chars: {stats['chars']} | Tokens: {stats['tokens']}")
        print(f"Status: {completion_status}")
        print("\n---------------------------------------------\n")

        results.append({
            "question": q,
            "answer": stats["text"],
            "time": stats["time"],
            "chars": stats["chars"],
            "tokens": stats["tokens"],
            "truncated": truncated,
            "completed": not truncated
        })

    return results


In [4]:
# Function to build a prompt in the instruction format expected by Mistral-Instruct models

def build_mistral_instruction_prompt(instruction: str, question: str):
    """
    Build a prompt using the instruction format expected by Mistral-Instruct models.
    This usually improves instruction-following and structure.
    """
    return f"<s>[INST] {instruction.strip()}\n\n{question.strip()} [/INST]"

In [5]:
# Function to run all prompt-engineering experiments on a list of questions

def run_all_experiments(llm, questions, experiment_configs):
    """
    Run all prompt experiments with:
    - Mistral instruction formatting
    - timing and length stats
    - graceful handling of interruptions
    - NO saving, NO checkpoints
    """

    all_results = {}

    for exp in experiment_configs:
        name = exp["name"]
        instruction = exp["instruction"]

        print(f"\n========== {name.upper()} ==========\n")

        all_results[name] = []

        for idx, q in enumerate(questions, 1):

            print(f"\n--- {name} | Q{idx}/{len(questions)} ---\nQ: {q}\n")

            # Build prompt
            prompt = build_mistral_instruction_prompt(instruction, q).lstrip("<s>").strip()

            try:
                stats = get_response(
                    llm,
                    prompt,
                    max_tokens=exp["max_tokens"],
                    temperature=exp["temperature"],
                    top_p=exp["top_p"],
                    top_k=exp["top_k"],
                    stop=["</s>", "\n\n\n"]
                )

                print("ANSWER:\n", stats["text"])
                print(f"\nTime: {stats['time']:.2f}s | Chars: {stats['chars']} | Tokens: {stats['tokens']}")
                print("-" * 50)

                result = {
                    "question": q,
                    "answer": stats["text"],
                    "time": stats["time"],
                    "chars": stats["chars"],
                    "tokens": stats["tokens"]
                }

            except Exception as e:
                print("\n--- INTERRUPTED ---")
                print(f"Error: {e}\n")

                result = {
                    "question": q,
                    "answer": "<INTERRUPTED>",
                    "time": None,
                    "chars": None,
                    "tokens": None
                }

                # Still raise so you see the error
                raise

            all_results[name].append(result)

    return all_results


In [6]:
# Function to build a comparison table from baseline and experiment results

def build_comparison_table(baseline_results, experiment_results):
    def avg(field, results):
        return round(sum(r[field] for r in results) / len(results), 2)

    def truncations(results):
        endings = (".", "!", "?")
        return sum(1 for r in results if not r["answer"].strip().endswith(endings))

    rows = []

    rows.append({
        "Experiment": "Baseline",
        "Avg Time (s)": avg("time", baseline_results),
        "Avg Chars": avg("chars", baseline_results),
        "Avg Tokens": avg("tokens", baseline_results),
        "Possible Truncations": truncations(baseline_results)
    })

    for name, results in experiment_results.items():
        rows.append({
            "Experiment": name,
            "Avg Time (s)": avg("time", results),
            "Avg Chars": avg("chars", results),
            "Avg Tokens": avg("tokens", results),
            "Possible Truncations": truncations(results)
        })

    return pd.DataFrame(rows)

---
## <span style="color:#87CEEB;">**Question Answering Using LLM (Test Question & Reponse)**</span>

### Downloading and Loading a test model

In [7]:
# Define the test model details
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [8]:
# Download the model from Hugging Face Hub and save it to a path

# Authenticate
login(token=hf_token)

local_model_path = os.path.join("models", model_basename)

try:
    if os.path.exists(local_model_path):
        print(f"Model already downloaded at: {local_model_path}")
        model_path = local_model_path
    else:
        print("Downloading model from Hugging Face...")
        model_path = hf_hub_download(
            repo_id=model_name_or_path,
            filename=model_basename,
            local_dir="models"
        )
        print(f"Model downloaded to: {model_path}")
except Exception as e:
    print("Error during model download/load:")
    print(str(e))
    raise


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


mistral-7b-instruct-v0.2.Q6_K.gguf:   0%|          | 0.00/5.94G [00:00<?, ?B/s]

Model downloaded to: models/mistral-7b-instruct-v0.2.Q6_K.gguf


In [9]:
# Test loading the model

print("Loading model...")

try:
    silent_buffer = io.StringIO()

    with redirect_stdout(silent_buffer), redirect_stderr(silent_buffer):
        llm = Llama(
            model_path=model_path,
            n_ctx=4096,
            n_batch=512,
            n_threads=max(1, (os.cpu_count() or 8) // 2),
            n_threads_batch=max(1, (os.cpu_count() or 8) // 2),
            n_gpu_layers=-1,
            verbose=False,
            logits_all=False,
            embedding=False
        )

    print("Model loaded successfully.\n")

except Exception as e:
    print("Failed to load model.")
    print(str(e))
    raise

Loading model...
Model loaded successfully.



### Test Response

In [10]:
response = get_response(llm, "What treatment options are available for managing hypertension?")
response

{'text': 'Hypertension, or high blood pressure, is a common condition that can increase the risk of various health problems, including heart disease, stroke, and kidney damage. The good news is that there are several treatment options available for managing hypertension, and the choice of treatment depends on the severity of the condition, underlying causes, and individual health factors. Here are some common treatment options for managing hypertension:\n\n1. Lifestyle modifications: Making lifestyle modifications is often the first line of treatment for managing hypertension. This may include adopting a healthy diet rich in fruits, vegetables, whole grains, and lean proteins, limiting sodium intake, reducing alcohol consumption, quitting smoking, and engaging in regular physical activity.\n2. Medications: If lifestyle modifications alone are not sufficient in controlling blood pressure, medications may be prescribed. There are several classes of medications used to treat hypertension,

**Observation**

- The model produces generally accurate and well-structured medical guidance, but responses are slow and can be incomplete, making it unreliable without further optimization and grounding.

---
## <span style="color:#87CEEB;">**Baseline QA with instruction-tuned LLM (Project Questions & Response)**</span>

In [11]:
# convert all questions into a list for easy access and processing
questions = [
    "What is the protocol for managing sepsis in a critical care unit?",
    "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
    "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
]


### Baseline Response (no prompt engineering)
- This captures the model’s natural behaviour. It shows how the model answers without structure, guidance, or constraints and gives a reference point to compare the engineered prompts against.

In [12]:
#  baseline experiment with no prompt engineering, just the question as input to the model and recording the response for each question.

baseline_results = run_baseline(llm, questions)


========== BASELINE (DEFAULT PARAMETERS) ==========


--- BASELINE QUESTION 1 of 5 ---

Q: What is the protocol for managing sepsis in a critical care unit?

ANSWER:

Sepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are the general steps for managing sepsis in a critical care unit:

1. Early recognition and suspicion: Septic patients may present with non-specific symptoms such as fever, chills, tachycardia, tachypnea, altered mental status, and lactic acidosis. It is essential to have a high index of suspicion for sepsis, especially in patients with known infections or risk factors.
2. Initial assessment and resuscitation: The first step in managing sepsis is to assess and resuscitate the patient. This includes assessing airway, breathing, circulation, and disability (ABCD) and providing appropriate interventions such as oxygen therapy, fluid resuscitation, and v

**Observation:**

- The baseline model uses the pretrained Mistral-7B-Instruct-v0.2 model in a direct question-answering setting without document retrieval. 

- It consistently produces clinically reasonable and structured responses, demonstrating strong general medical knowledge, but answers are often incomplete and lack source grounding. 
- Latency is relatively high (~12–13s per query), limiting real-time usability in clinical settings. This highlights the need for RAG to improve response reliability, completeness, and trustworthiness while maintaining or reducing response time.

---
## <span style="color:#87CEEB;">**Baseline QA with instruction-tuned LLM with Prompt Engineering (Project Questions & Response)**</span>

Prompt engineering was tested using the same Mistral-7B-Instruct model, with decoding parameters largely fixed to the baseline for fairness; only max_tokens was increased where necessary to prevent truncation in more structured response formats.

**Purpose:**
Test whether better prompting alone improves clarity, structure, and completeness before introducing retrieval with FAISS.

### LLM prompt Parameters

In [ ]:
# Prompt‑engineering experiments to evaluate different instruction styles
# using a consistent Mistral‑Instruct format:
# <s>[INST] <<SYS>> {system_instruction} <</SYS>> {question} [/INST]

# Fair comparison principles:
# - All experiments use the same Mistral‑7B‑Instruct model.
# - Only the instruction text changes; the model weights remain untouched.
# - temperature, top_p, and top_k stay identical across experiments so differences reflect prompt design, not sampling randomness.
# - max_tokens is adjusted only when a format requires more space.

# Baseline reference:
#   max_tokens = 128
#   temperature = 0.0
#   top_p = 0.95
#   top_k = 50

experiment_configs = [
    {
        "name": "Clinical Guideline Mode",
        # Rationale:
        # - Exactly 4 bullets keeps output short and comparable.
        # - One sentence per bullet prevents drift or expansion.
        # - No intro or conclusion removes token waste.
        "instruction": (
            "You are a medical knowledge assistant. "
            "Answer in exactly 4 short bullet points. "
            "Each bullet must be one complete sentence. "
            "Focus only on key clinical guidance. "
            "No introduction and no conclusion."
        ),
        "temperature": 0.0,
        "top_p": 0.95,
        "top_k": 50,
        "max_tokens": 96
    },
    {
        "name": "Explain Like I'm 15",
        # Rationale:
        # - Tests clarity and simplification.
        # - Exactly 3 short sentences keeps output tight and fast.
        "instruction": (
            "You are a medical knowledge assistant. "
            "Explain this in exactly 3 short sentences for a 15-year-old. "
            "Avoid jargon but keep the meaning medically correct. "
            "No introduction and no conclusion."
        ),
        "temperature": 0.0,
        "top_p": 0.95,
        "top_k": 50,
        "max_tokens": 96
    },
    {
        "name": "Structured Medical Framework",
        # Rationale:
        # - Highly evaluation-friendly.
        # - Fixed 4-section structure with one sentence each.
        "instruction": (
            "You are a medical knowledge assistant. "
            "Answer using exactly these 4 sections: "
            "1. Definition "
            "2. Key symptoms "
            "3. Causes "
            "4. General management principles. "
            "Write exactly one short sentence per section. "
            "No introduction and no conclusion."
        ),
        "temperature": 0.0,
        "top_p": 0.95,
        "top_k": 50,
        "max_tokens": 120
    },
    {
        "name": "Expert Deep-Dive Mode",
        # Rationale:
        # - Expert tone but tightly constrained.
        # - Exactly 5 concise sentences avoids uncontrolled expansion.
        "instruction": (
            "You are a medical knowledge assistant. "
            "Provide an expert-level answer in exactly 5 concise sentences. "
            "Prioritise mechanism, key clinical features, and management principles. "
            "Do not add filler."
        ),
        "temperature": 0.0,
        "top_p": 0.95,
        "top_k": 50,
        "max_tokens": 144
    },
    {
        "name": "Bullet-Point Precision Mode",
        # Rationale:
        # - Compact factual mode.
        # - Exactly 4 bullets ensures consistency and speed.
        "instruction": (
            "You are a medical knowledge assistant. "
            "Answer in exactly 4 bullet points. "
            "Each bullet must contain one important clinical fact in one sentence. "
            "Do not repeat ideas."
        ),
        "temperature": 0.0,
        "top_p": 0.95,
        "top_k": 50,
        "max_tokens": 96
    }
]


In [14]:
experiment_results = run_all_experiments(llm, questions, experiment_configs)


========== CLINICAL GUIDELINE MODE ==========


--- Clinical Guideline Mode | Q1/5 ---
Q: What is the protocol for managing sepsis in a critical care unit?

ANSWER:
 1. Initiate early and aggressive fluid resuscitation to maintain adequate tissue perfusion and organ function.
2. Administer broad-spectrum antibiotics as soon as possible based on suspected infection source and local microbiology data.
3. Provide adequate oxygenation and ventilatory support if the patient develops respiratory failure.
4. Implement hemodynamic support, such as vasopressors or inotropes, if needed

Time: 5.30s | Chars: 416 | Tokens: 56
--------------------------------------------------

--- Clinical Guideline Mode | Q2/5 ---
Q: What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

ANSWER:
 1. Appendicitis is characterized by abdominal pain, usually located in the lower right quadrant, often accompanied by loss of app

**Observation**

- Prompt engineering significantly improves response structure, consistency, and speed (reducing latency from ~13s to ~4–7s), but strict constraints introduce truncation and loss of completeness across multiple modes. 

- Simpler formats (Explain Like I’m 15, Clinical Guideline) offer the best balance of clarity and efficiency, while more rigid or detailed modes risk incomplete or cut-off answers. This demonstrates that prompt design alone can enhance usability, but still requires RAG to ensure completeness, accuracy, and clinical reliability.

- This helps identify which prompting strategy produces the most stable and concise answers before introducing the FAISS-based RAG pipeline. **A good takeaway is that the RAG system prompt should combine the strengths of Clinical Guideline Mode and Explain Like I’m 15: clinically focused, simple, and structured, but flexible enough to fully answer the question.**

### Experiment Results Interpreted Against Baseline

In [15]:
# Build and print the comparison table summarising all results

comparison_df = build_comparison_table(baseline_results, experiment_results)
comparison_df


,Experiment,Avg Time (s),Avg Chars,Avg Tokens,Possible Truncations
0,Baseline,12.81,1051.8,167.2,5
1,Clinical Guideline Mode,5.08,413.8,57.8,5
2,Explain Like I'm 15,4.76,387.0,64.4,2
3,Structured Medical Framework,5.44,452.4,64.0,1
4,Expert Deep-Dive Mode,7.35,604.8,84.8,5
5,Bullet-Point Precision Mode,4.98,409.6,60.4,4


**Report Summary Table**

| Experiment                       | Speed            | Output Quality      | Completeness             | RAG Suitability | Key Insight                                       |
| -------------------------------- | ------------------ | --------------------- | --------------------------- | ------------------ | ------------------------------------------------- |
| **Baseline**                     | Slow (12.8s)     | High but verbose      |  Poor (5 truncations)      |  Low              | Too slow, unstructured, and unreliable            |
| **Clinical Guideline Mode**      |  Fast (5.2s)      | Structured & clinical |  Poor (5 truncations)      |  Medium          | Good format but overly restrictive → truncation   |
| **Explain Like I’m 15**          |  Fastest (4.7s)   | Clear & simple        |  Moderate (2 truncations) |  High             | Best balance of clarity, speed, and flexibility   |
| **Structured Medical Framework** |  Moderate (5.4s) | Highly organized      |  Best (1 truncation)       |  Medium          | Most consistent but too rigid for diverse queries |
| **Expert Deep-Dive Mode**        |  Slower (7.5s)    | Detailed & rich       |  Poor (5 truncations)      |  Low              | Too verbose → high truncation + latency           |
| **Bullet-Point Precision Mode**  |  Fast (5.0s)      | Concise & factual     |  Poor (4 truncations)      |  Medium          | Efficient but loses completeness                  |



---
## <span style="color:#87CEEB;">**RAG Pipeline Implementation → FAISS → Retrieval → Augmented Prompt → Answer Generation.**</span>


### RAG Pipeline Workflow & Summary

- This stage transforms raw medical content into structured, retrievable knowledge, enabling the model to generate accurate, context-grounded responses by using relevant information from a document database.

- The RAG pipeline enhances LLM performance by combining structured retrieval with controlled generation, delivering faster, more accurate, and context-grounded medical responses.


| Stage                       | Process                                                | Purpose                                           | Key Outcome                          |
| --------------------------- | ------------------------------------------------------ | ------------------------------------------------- | ------------------------------------ |
| **1. Data Preparation**     | Load, clean and preprocess medical manual                    | Remove noise while preserving clinical meaning    | High-quality, usable text            |
| **2. Chunking**             | Split text into smaller sections with overlap          | Improve retrieval precision and context relevance | Meaningful, searchable chunks        |
| **3. Embedding Generation** | Convert chunks into vector embeddings (MPS-enabled)    | Enable semantic search                            | Faster, scalable similarity matching |
| **4. Vector Storage**       | Store embeddings in FAISS index                        | Allow efficient retrieval of relevant information | Persistent, reusable knowledge base  |
| **5. Retrieval (MMR)**      | Fetch top-k diverse and relevant chunks                | Reduce redundancy and improve coverage            | High-quality contextual input        |
| **6. Prompt + Generation**  | Combine retrieved context with system prompt and query | Generate grounded, structured responses           | Accurate, context-aware answers      |
| **7. Evaluation**           | Assess groundedness and relevance                      | Ensure reliability and trustworthiness            | Measurable response quality          |


### Loading the Data

In [16]:
# Loading data

try:
    # Dynamically select the parent directory
    parent_dir = Path.cwd() # Get the directory where the current script is located

    # Define paths for models, index, and manual PDF
    models_dir = parent_dir / "models"
    index_path = models_dir / "medical_manual_faiss_index"
    manual_pdf_path = parent_dir / "medical_diagnosis_manual.pdf"

    # Ensure that the models directory exists
    models_dir.mkdir(parents=True, exist_ok=True)

    # Load the PDF manually
    pdf_loader = PyMuPDFLoader(str(manual_pdf_path))
    manual = pdf_loader.load()

    # If no error occurred, print success message
    print("Load successful.")

except Exception as e:
    # Catch any exception that occurs and display the error message
    print(f"Error occurred: {str(e)}")

Load successful.


### Data Overview

-  Checking the first 5 pages and manual length

In [17]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")

print(f"\nLength of manual: {len(manual)}")

Page Number : 1
omotayo@outlook.com
EGVD5P09O3
This file is meant for personal use by omotayo@outlook.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
omotayo@outlook.com
EGVD5P09O3
This file is meant for personal use by omotayo@outlook.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .......................................................................................................................................

**Observation**

- The medical manual is a **comprehensive, multi-specialty clinical reference (4114 pages)** covering a wide range of systems and conditions, making it a strong and reliable knowledge base for RAG. However, the presence of **repeated headers, legal disclaimers, and structural noise** highlights the need for thorough preprocessing to ensure clean, relevant retrieval. 

- Overall, its breadth supports diverse clinical queries, but effective chunking and filtering are critical to maximize retrieval accuracy and model performance.


### Data Cleaning and Normalisation

A light, conservative data cleaning step was applied to remove noise (e.g., watermarks, headers, and formatting artefacts) while preserving medically relevant content and document structure, ensuring accuracy in downstream chunking and retrieval.

In [18]:
# Utility function to clean the manual text by removing noise and irrelevant content while preserving important medical information for better processing in the RAG pipeline.

def clean_manual_docs(documents):
    """
    Light cleaning for RAG preparation.

    Keeps most page content intact and removes only obvious noise:
    - watermark/legal notice lines
    - email addresses
    - ID-like codes
    - standalone page numbers
    - hyphenated line breaks
    - excess whitespace
    """

    cleaned_docs = []

    for doc in documents:
        # avoid mutating the original docs
        new_doc = deepcopy(doc)
        text = new_doc.page_content

        # Remove email addresses
        text = re.sub(r"\b\S+@\S+\b", " ", text, flags=re.IGNORECASE)

        # Remove short uppercase/alphanumeric watermark codes like EGVD5P09O3
        text = re.sub(r"\b[A-Z0-9]{8,}\b", " ", text)

        # Remove watermark / legal notice lines
        text = re.sub(r"(?im)^.*personal use.*$", " ", text)
        text = re.sub(r"(?im)^.*sharing or publishing.*$", " ", text)
        text = re.sub(r"(?im)^.*legal action.*$", " ", text)

        # Remove standalone page numbers only
        text = re.sub(r"(?m)^\s*\d+\s*$", " ", text)

        # Fix hyphenated line breaks: e.g. treat-\nment -> treatment
        text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)

        # Remove excessive dot leaders, but don't destroy normal punctuation
        text = re.sub(r"\.{4,}", " ", text)

        # Normalize whitespace
        text = re.sub(r"\r", "\n", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        text = re.sub(r"[ \t]+", " ", text)
        text = text.strip()

        # Keep page if it still has meaningful content
        if len(text) > 50:
            new_doc.page_content = text
            cleaned_docs.append(new_doc)

    return cleaned_docs

In [19]:
cleaned_manual = clean_manual_docs(manual)
print(f"Original pages: {len(manual)}")
print(f"Cleaned pages: {len(cleaned_manual)}")

# inspect the first few pages of the cleaned manual to verify cleaning results and ensure that important content is preserved while unwanted elements are removed.
for i in range(min(5, len(cleaned_manual))):
    print(f"\n--- Cleaned Page {i} ---\n")
    print(cleaned_manual[i].page_content[:500])

Original pages: 4114
Cleaned pages: 4112

--- Cleaned Page 0 ---

Table of Contents
 
Front 
 
Cover 
 
Front Matter 
 
1 - Nutritional Disorders 
 
Chapter 1. Nutrition: General Considerations 
 
Chapter 2. Undernutrition 
 
Chapter 3. Nutritional Support 
 
Chapter 4. Vitamin Deficiency, Dependency & Toxicity 
 
Chapter 5. Mineral Deficiency & Toxicity 
 
Chapter 6. Obesity & the Metabolic Syndrome 
 
2 - Gastrointestinal Disorders 
 
Chapter 7. Approach to the Patient With Upper GI Complaints 
 
Chapter 8. Approach to the Patient With Lower GI Complaints 
 

--- Cleaned Page 1 ---

Chapter 44. Foot & Ankle Disorders 
 
Chapter 45. Tumors of Bones & Joints 
 
5 - Ear, Nose, Throat & Dental Disorders 
 
Chapter 46. Approach to the Patient With Ear Problems 
 
Chapter 47. Hearing Loss 
 
Chapter 48. Inner Ear Disorders 
 
Chapter 49. Middle Ear & Tympanic Membrane Disorders 
 
Chapter 50. External Ear Disorders 
 
Chapter 51. Approach to the Patient With Nasal & Pharyngeal Symptoms 
 


**Observation**

- The cleaning process had minimal impact on data volume (4114 → 4112 pages), indicating that noise removal was effective without losing meaningful medical content. 

- The preserved structured layout (chapters, sections, systems) confirms that document integrity was maintained, which is critical for accurate chunking and retrieval. 
- Overall, the dataset is now cleaner while retaining its hierarchical organization, improving the quality and relevance of downstream RAG retrieval.


### Data Chunking

In [28]:
# Chunking the manual into smaller pieces for better processing and retrieval

splitter = RecursiveCharacterTextSplitter(
    chunk_size=650,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(cleaned_manual)

# Keep page metadata visible for source tracing during retrieval
for chunk in chunks:
    chunk.metadata["page_num"] = chunk.metadata.get("page") if "page" in chunk.metadata else chunk.metadata.get("page_num")

print(f"Total chunks: {len(chunks)}\n")
print(f"First chunk:\n{chunks[0].page_content}")



Total chunks: 26590

First chunk:
Table of Contents
 
Front 
 
Cover 
 
Front Matter 
 
1 - Nutritional Disorders 
 
Chapter 1. Nutrition: General Considerations 
 
Chapter 2. Undernutrition 
 
Chapter 3. Nutritional Support 
 
Chapter 4. Vitamin Deficiency, Dependency & Toxicity 
 
Chapter 5. Mineral Deficiency & Toxicity 
 
Chapter 6. Obesity & the Metabolic Syndrome 
 
2 - Gastrointestinal Disorders 
 
Chapter 7. Approach to the Patient With Upper GI Complaints 
 
Chapter 8. Approach to the Patient With Lower GI Complaints 
 
Chapter 9. Diagnostic & Therapeutic GI Procedures 
 
Chapter 10. GI Bleeding 
 
Chapter 11. Acute Abdomen & Surgical Gastroenterology


**Observation**

- The dataset was transformed into 26,590 well-sized chunks, improving contextual continuity and enabling more reliable fine-grained retrieval across the medical corpus.
- Retaining the TOC adds valuable **high-level structural context**, which can support navigation and broad query understanding. 

- Overall, the chunking preserves both detail and document hierarchy, strengthening downstream RAG performance.


In [29]:
# Verify chunk overlap to ensure context continuity across boundaries

num_preview = 5
print(f"Total chunks: {len(chunks)}\n")

for i in range(num_preview - 1):
    a = chunks[i].page_content
    b = chunks[i+1].page_content

    overlap = a[-150:]
    similarity = overlap[:80] in b or overlap[-80:] in b

    print(f"Chunk {i} → {i+1} | Overlap Likely OK: {similarity}")


# Preview the similarity between adjacent chunks to check for excessive overlap or redundancy

chunk_lengths = [len(c.page_content) for c in chunks]

print(f"\nMin chunk: {min(chunk_lengths)}")
print(f"Max chunk: {max(chunk_lengths)}")
print(f"Avg chunk: {sum(chunk_lengths)/len(chunk_lengths)}")

Total chunks: 26590

Chunk 0 → 1 | Overlap Likely OK: True
Chunk 1 → 2 | Overlap Likely OK: True
Chunk 2 → 3 | Overlap Likely OK: True
Chunk 3 → 4 | Overlap Likely OK: False

Min chunk: 55
Max chunk: 650
Avg chunk: 572.4276043625423


**Observation**

- With overlap now largely preserved and average chunk size remaining compact, the chunking strategy is better aligned for accurate and context-aware RAG performance.

*Chunking Analysis Summary*

| Insight Category            | Observation                                     | Interpretation                                                      | Implication for RAG                                               |
| --------------------------- | ----------------------------------------------- | ------------------------------------------------------------------- | ----------------------------------------------------------------- |
| **Total Chunks**            | 26,590 chunks generated                         | Large corpus successfully segmented into fine-grained units         | Enables scalable and precise retrieval                            |
| **Chunk Size Distribution** | Min: 55, Max: 650, Avg: ~572                    | Consistent chunk sizing close to target, with limited fragmentation | Balanced trade-off between context depth and retrieval efficiency |
| **Overlap Behaviour**       | Majority of adjacent chunks show overlap (True) | Overlap is largely preserved across chunks                          | Improves contextual continuity between segments                   |
| **Overlap Variability**     | Occasional False overlaps                       | Due to natural text boundary splitting (recursive strategy)         | Expected behaviour; minimal impact on semantic coherence          |
| **Small Chunks**            | Few chunks near lower bound (~55 chars)         | Likely structural elements (headings, TOC, short sections)          | Acceptable, especially with TOC retained for structural context   |
| **Overall Chunk Quality**   | High average size with consistent distribution  | Chunks contain meaningful, information-dense medical content        | Suitable for embedding and FAISS-based retrieval                  |


### Embedding

In [31]:
# Prefer MPS acceleration for embeddings; default to CPU if unavailable

try:
        embedding_device = "mps" if torch.backends.mps.is_available() else "cpu"
        print(f"Embedding device: {embedding_device}")

        embedding_model = HuggingFaceBgeEmbeddings(
            model_name="BAAI/bge-small-en-v1.5",
            model_kwargs={"device": embedding_device},
            encode_kwargs={
                "normalize_embeddings": True,
                "batch_size": 32
            }
        )

        print("Embedding model loaded successfully.")

except Exception as e:
        print("Error loading embedding model:")
        print(str(e))
        raise


Embedding device: mps


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [32]:
# Verify embedding dimensions are consistent across chunks for FAISS compatibility

embedding_1 = embedding_model.embed_query(chunks[0].page_content)
embedding_2 = embedding_model.embed_query(chunks[1].page_content)

print("Dimension of the embedding vector ",len(embedding_1))
print(f"Are the dimensions equal? {len(embedding_1) == len(embedding_2)}")

Dimension of the embedding vector  384
Are the dimensions equal? True


**Observations**

- The embedding model produces consistent 384-dimensional vectors, confirming compatibility and stability for FAISS indexing and retrieval. 

- This ensures reliable similarity comparisons and prevents errors during vector storage and search operations.


### Vector Database and Indexing

- The generated embeddings were stored in a FAISS vector index to enable fast, similarity-based retrieval. The index was persisted locally after creation, allowing the system to reuse precomputed embeddings without recomputation on subsequent runs. This significantly improves performance, reduces processing time, and ensures a more responsive and scalable medical question-answering application.


In [33]:
# Build the FAISS vector store once, then reuse the saved index on later runs.

if index_path.exists():
    print(f"Loading cached FAISS index from: {index_path}")
    vectorstore = FAISS.load_local(
        str(index_path),
        embedding_model,
        allow_dangerous_deserialization=True
    )
else:
    print("Building FAISS index from document chunks...")
    vectorstore = FAISS.from_documents(chunks, embedding_model)
    vectorstore.save_local(str(index_path))
    print(f"Saved FAISS index to: {index_path}")


Building FAISS index from document chunks...
Saved FAISS index to: /Users/Probook/Library/CloudStorage/OneDrive-Personal/Projects_Portfolio/Project5-NLP_-Generative AI-Medical Assistant/models/medical_manual_faiss_index


In [34]:
# Run test queries to verify retrieval relevance from the vector store

results = vectorstore.similarity_search("What is the protocol for managing sepsis?", k=3)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---\n")
    print(doc.page_content[:500])



--- Result 1 ---

can occur. Treatment is aggressive fluid resuscitation, antibiotics, surgical excision of infected
or necrotic tissues and drainage of pus, supportive care, and sometimes intensive control of
blood glucose and administration of corticosteroids and activated protein C.
A spectrum of severity exists (see
Table 227-1).
Sepsis is infection accompanied by an acute inflammatory reaction with systemic manifestations
associated with release into the bloodstream of numerous endogenous mediators of inflam

--- Result 2 ---

Parenteral antibiotics should be given after specimens of blood, body fluids, and wound sites have been
taken for Gram stain and culture. Very prompt empiric therapy, started immediately after suspecting
sepsis, is essential and may be lifesaving. Antibiotic selection requires an educated guess based on the
suspected source, clinical setting, knowledge or suspicion of causative organisms and of sensitivity
patterns common to that specific inpatient unit, an

**Observation**

- The retrieval results are highly relevant and clinically aligned with the query, capturing key aspects of sepsis including definition, management, and severity classification. The diversity across chunks (treatment, antibiotics, severity) indicates effective MMR-based retrieval. 

**Overall, the vector store is successfully returning meaningful, context-rich information for grounded response generation.**


In [35]:
# Verify the FAISS vector store is ready for retrieval.
vectorstore

The FAISS vector store is successfully initialized and loaded in memory, confirming that the embedding index is ready for efficient similarity-based retrieval.


### Retriever

A retriever was configured on the FAISS index to perform top-k similarity search, enabling retrieval of the most relevant medical context for each query. A value of **k = 4** was selected to balance completeness and precision—providing sufficient context for accurate, grounded responses while minimising noise from irrelevant information.


In [36]:
# Set up a retriever that prefers diverse but relevant chunks.
default_k = 4

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": default_k,
        "fetch_k": 10,
        "lambda_mult": 0.7
    }
)

In [37]:
# testing the retriever

query = "What is the protocol for managing sepsis in a critical care unit?"

# Fix: Use .invoke() for retrieving documents, as get_relevant_documents might not be available.
retrieved_docs = retriever.invoke(query)

print(f"Total retrieved chunks: {len(retrieved_docs)}")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Retrieved Chunk {i} ---\n")
    print(doc.page_content[:700])

Total retrieved chunks: 4

--- Retrieved Chunk 1 ---

16 - Critical Care Medicine
Chapter 222. Approach to the Critically Ill Patient
Introduction
Critical care medicine specializes in caring for the most seriously ill patients. These patients are best
treated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special
populations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high
nurse:patient ratio to provide the necessary high intensity of service, including treatment and monitoring
of physiologic parameters.
Supportive care for the ICU patient includes provision of adequate nutrition (see p. 21) and prevention of

--- Retrieved Chunk 2 ---

Parenteral antibiotics should be given after specimens of blood, body fluids, and wound sites have been
taken for Gram stain and culture. Very prompt empiric therapy, started immediately after suspecting
sepsis, is essential and may be lifesaving. Antibiotic selection require

**Observation**

The retriever was configured with MMR and k = 4, which returned relevant, diverse, and clinically meaningful chunks covering key aspects of sepsis, including definition, severity, treatment, and ICU management. While this setting provides strong contextual coverage aligned with the project objectives, the inclusion of broader ICU content indicates a minor precision trade-off, highlighting the potential for further optimisation through parameter tuning or reranking.


### System and User Prompt Template

- The system prompt defines how the model behaves in the RAG pipeline, ensuring responses are based only on retrieved medical context. This reduces hallucinations and improves reliability, which is critical in healthcare settings. 

- It also enforces clear and concise answers suitable for clinical decision support.


In [38]:
# Step 1 — Create instruction (system prompt)
# The system prompt balances structure, flexibility, and grounding, ensuring complete, context-based, and clinically reliable responses without the truncation issues observed in earlier prompt experiments.

qna_system_message = """
You are a medical knowledge assistant using trusted clinical reference material.

Your task is to answer the question using ONLY the provided context.

Guidelines:
- Provide **3 to 5 concise bullet points**.
- Ensure each bullet is one clear, medically accurate sentence.
- Do not exceed 5 bullet points, or the response will be truncated.
- Ensure the answer is **complete** — do not leave points unfinished.
- Cover all key aspects present in the context without omitting important details.
- Prioritize key clinical guidance, including symptoms, causes, and management where relevant.
- Use simple, clear language while maintaining medical correctness.
- Avoid unnecessary detail or repetition.

Safety rules:
- If the context does not contain sufficient information, respond with:
  "Insufficient information in the provided context."
- Do not use external knowledge.
- Do not guess or hallucinate.

Do not include introductions or conclusions — go straight to the answer.
"""

In [39]:
# Step 2 — Inject context into the question (user prompt)

qna_user_message_template = """
Context:
{context}

Question:
{question}

Answer:
"""


### Response Function

In [40]:
# Function to generate a RAG response by integrating retrieval and generation

def generate_rag_response(
    user_input,
    k=4,
    max_tokens=192,
    temperature=0,
    top_p=0.9,
    top_k=40
):
    """
    Generates a RAG-based response by integrating document retrieval and generation.

    1. Retrieves the top-k relevant document chunks using FAISS.
    2. Constructs a context-aware prompt by formatting the retrieved chunks.
    3. Passes the prompt to the Mistral-Instruct model to generate a response.
    
    Parameters:
        user_input (str): The query or question from the user.
        k (int): The number of chunks to retrieve (default is 4).
        max_tokens (int): The maximum number of tokens for the generated response.
        temperature (float): Controls the randomness of the output (default is 0).
        top_p (float): Controls nucleus sampling for output generation.
        top_k (int): Limits the number of top tokens to consider for generation.

    Returns:
        str: The generated response based on the input and retrieved context.
    """
    
    # Initialize the retriever with MMR and set retrieval parameters
    local_retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": k,
            "fetch_k": max(10, 2 * k),  # Fetch more documents for better coverage
            "lambda_mult": 0.7  # Adjusts diversity
        }
    )

    # Retrieve relevant documents based on the user input
    docs = local_retriever.invoke(user_input)

    # Prepare context by extracting page numbers and content from retrieved docs
    context_parts = []
    for doc in docs:
        page_num = doc.metadata.get("page_num", doc.metadata.get("page", "unknown"))
        context_parts.append(f"[Page {page_num}]\n{doc.page_content}")

    context = "\n\n".join(context_parts)  # Combine all context parts

    # Format the user message using the retrieved context
    user_message = qna_user_message_template.format(
        context=context,
        question=user_input
    )

    # Construct the final prompt by embedding the system message and user input
    prompt = f"[INST] {qna_system_message}\n\n{user_message} [/INST]"

    try:
        # Generate the response using the language model
        output = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=["</s>"]  # Ensure proper termination of the response
        )
        response = output["choices"][0]["text"].strip()  # Extract the generated text

        # Ensure response has no more than 5 bullet points
        # Split by bullet points (assuming each starts with "*") and limit to 5
        bullet_points = response.split("*")
        bullet_points = [point.strip() for point in bullet_points if point.strip()]  # Remove empty entries
        response = "\n".join([f"* {point}" for point in bullet_points[:5]])  # Limit to 5 bullet points

    except Exception as e:
        # Handle errors in case of failure during generation
        response = f"Error: {e}"

    return response


---
## <span style="color:#87CEEB;">**Question Answering using RAG**

In [41]:
# Quick QA run using the updated retrieval and generation settings.

for i, q in enumerate(questions, 1):
    print(f"\n--- QUESTION {i} ---")
    print("Q:", q)
    print("\nANSWER:\n")
    print(generate_rag_response(q))
    print("\n" + "-" * 80)



--- QUESTION 1 ---
Q: What is the protocol for managing sepsis in a critical care unit?

ANSWER:

* Monitor the critically ill sepsis patient in an ICU with experienced personnel.
* Provide adequate nutrition and prevent infections.
* For septic shock, give parenteral antibiotics after taking specimens for culture and start prompt empiric therapy.
* Monitor physiologic parameters, including systemic pressure, CVP or PAOP, pulse oximetry, ABGs, blood glucose, lactate, electrolyte levels, renal function, and possibly sublingual PCO2.
* Administer replacement-dose corticosteroids.

--------------------------------------------------------------------------------

--- QUESTION 2 ---
Q: What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

ANSWER:

* Common symptoms of appendicitis include epigastric or periumbilical pain followed by right lower quadrant abdominal pain, nausea, vomiting, and anorexia.

**Insight From Questions:**

The system effectively uses the **retrieved medical context** to generate **grounded responses** for various clinical questions, as seen in the results:

1. **Sepsis Management in ICU:** Clear sepsis management steps with ICU care, antibiotics, fluid resuscitation, and corticosteroids, but minor drift into broader ICU context

2. **Appendicitis Symptoms & Surgery:** Accurate symptom description and surgical treatment (appendectomy), with a bit too much detail on surgical nuances.

3. **Treatment for Sudden Patchy Hair Loss:** Comprehensive treatments for alopecia areata, traction alopecia, tinea capitis, and trichotillomania, with some minor extraneous details.

4. **Brain Injury Treatment:** Focused on emergency care, surgery, and rehabilitation, but includes generic coma management tips not directly related to the query.

5. **Fractured Leg Treatment:** Detailed care instructions, including leg elevation, compression, and anticoagulation therapy, with minor repetitive information.


**Insight from RAG Response (Without Fine-Tuning)**

* **Relevance:** The answers provide relevant and complete clinical information, but some minor irrelevant details (e.g., general ICU care or extra treatment nuances) slightly dilute the focus.

* **Semantic Precision:** The content aligns well with the queries, though some responses could be more specific, especially for conditions like appendicitis and brain injury.
* **Contextual Completeness:** The answers cover essential clinical steps, though occasionally general context (e.g., ICU care) is included when not directly needed.
* **Improvement Areas:** Reducing redundant information and refining top-k retrieval could enhance the focus and precision of the answers.



### Fine-Tuning (Parameter-Based)

**Fine-tuning was performed by adjusting the retrieval and generation parameters to improve the focus and relevance of the answers**. 

| Parameter     | Before | After | Rationale & Effect                                                                              |
| ------------- | ------ | ----- | ----------------------------------------------------------------------------------------------- |
| `k`           | 4      | 3     | Reduced to optimize context precision and avoid overloading retrieval with less relevant chunks |
| `max_tokens`  | 192    | 200   | Increased to allow for more detailed responses while maintaining token efficiency               |
| `temperature` | 0      | 0     | Ensures deterministic output, suitable for clinical decision-making |
| `top_p`       | 0.9    | 0.9   | No change—ensures balanced diversity while avoiding irrelevant content      |
| `top_k`       | 40     | 40    | No change—maintains the pool of top tokens to prevent truncation in generation            |



In [42]:
# Tuned QA run with explicit parameters.

for i, q in enumerate(questions, 1):
    print(f"\n--- QUESTION {i} ---")
    print("Q:", q)
    print("\nANSWER:\n")
    print(generate_rag_response(q,  
        k=3,
        max_tokens=200))
    print("\n" + "-" * 80)


--- QUESTION 1 ---
Q: What is the protocol for managing sepsis in a critical care unit?

ANSWER:

* 1. Initiate very prompt empiric antibiotic therapy after taking specimens for culture and Gram stain.
2. Monitor systemic pressure, CVP or PAOP, pulse oximetry, ABGs, blood glucose, lactate, electrolyte levels, renal function, and sublingual PCO2.
3. Measure urine output using an indwelling catheter.
4. Provide replacement-dose corticosteroids for patients with septic shock.
5. Perform fluid resuscitation with 0.9% saline until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg.
6. Oliguria with hypotension does not contraindicate vigorous fluid resuscitation.

--------------------------------------------------------------------------------

--- QUESTION 2 ---
Q: What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

ANSWER:

* Appendicitis is characterized by abdominal pain, anorexia, 

**Key Insight:**

* **Sepsis Management**: The response provides a clear and comprehensive protocol for managing sepsis, covering important clinical guidelines like monitoring, antibiotic therapy, fluid resuscitation, and corticosteroid use. The concise bullet-point format effectively communicates key actions required in the ICU for sepsis management, ensuring clarity without unnecessary details.

* **Appendicitis Symptoms and Treatment**: The response covers the essential symptoms of appendicitis, including pain characteristics and gastrointestinal symptoms, and details surgical treatment options. It also mentions the acceptable negative appendectomy rate and identifies contraindications like inflammatory bowel disease, which is critical for clinical decision-making.

* **Hair Loss Treatment**: The answer outlines treatment options for alopecia areata, including topical corticosteroids and immunotherapy, while also addressing other causes like traction alopecia and tinea capitis. The inclusion of treatments like oral antifungals and behavior modification adds important context for non-alopecia-related causes.

* **Traumatic Brain Injury (TBI) Management**: The response highlights key treatments for TBI, emphasizing the importance of securing the airway, managing intracranial pressure, and starting cognitive rehabilitation early. It also distinguishes between complete and partial spinal cord injuries, providing actionable insights for tailored care.

* **Fracture Treatment**: The answer provides practical treatment steps for a leg fracture, such as elevating the injured leg, applying compression, and using ice. It also includes critical considerations for preventing thrombosis, which is essential for patients with fractures, particularly those immobilized for extended periods.


**Key Insights:**

* The generated responses maintain clinical precision and clarity, adhering to the system prompt's requirements for concise, medically accurate, and contextually grounded answers.
* **Improved completeness**: The increase in **`max_tokens`** allowed for more comprehensive responses without truncating key clinical details.
* **Response consistency**: The **`temperature`** parameter at 0 ensured deterministic, repeatable outputs, crucial for clinical applications.
* **Optimized retrieval**: Adjusting **`k`** to 3 ensured more focused and relevant context was retrieved, improving the overall precision of the answers.

**Next Steps:**

The output quality from fine-tuning with these parameters meets expectations for RAG-based question-answering systems. **For further refinement**, experimenting with **reranking/filtering** mechanisms and **post-processing** might help improve precision for some queries by eliminating less relevant chunks.


---
## <span style="color:#87CEEB;">**Output Evaluation**</span>

The **RAG system** uses two primary evaluators to ensure the quality of generated answers:

1. **Groundedness Rater**: Assesses whether the answer is fully supported by the retrieved context. The focus is on ensuring **accuracy** and **reliability** by verifying that no hallucinated or unsupported information is included.

2. **Relevance Rater**: Evaluates if the answer directly addresses the user's question, ensuring the **response is aligned** with the query and does not stray into unrelated details.

These evaluations are conducted using the **Mistral model** as both a **retriever** and **judge**, providing an automated quality check aligned with the project’s objectives of improving **reliable, context-grounded responses** for **clinical decision support**.

In [43]:
# System message for the groundedness rater to evaluate if the answer is fully supported by the context and provide a score with a brief explanation.

groundedness_rater_system_message = """
You are an evaluator.

Check whether the answer is fully supported by the provided context. The answer must be derived solely from the context and should not introduce any unsupported information.

Score:
- 1 = Not grounded (hallucinated or unsupported)
- 2 = Partially grounded
- 3 = Fully grounded

Return only the score and a concise explanation of why the answer is or isn't grounded.
"""

In [44]:
# System message for the relevance rater to assess if the answer directly addresses the question and provide a score with a brief explanation.

relevance_rater_system_message = """
You are an evaluator.

Check whether the answer directly addresses the question, focusing on the key elements of the query and ensuring the response is on-topic.

Score:
- 1 = Not relevant (fails to address the question or includes irrelevant content)
- 2 = Partially relevant (addresses part of the question but omits critical elements)
- 3 = Fully relevant (directly answers the question with no unnecessary information)

Return only the score and a brief explanation of why the answer is or isn't relevant.
"""

In [45]:
# User message template for the raters, which includes the context, question, and answer to be evaluated.

user_message_template = """
Context:
{context}

Question:
{question}

Answer:
{answer}
"""

In [ ]:
# Function to generate responses for groundedness and relevance evaluation by retrieving relevant document chunks

def generate_ground_relevance_response(
    user_input,
    k=3,
    max_tokens=200,
    temperature=0,
    top_p=0.9,
    top_k=40
):
    """
    Generate a RAG-based answer and evaluate it for groundedness and relevance.

    Workflow:
    1. Retrieve top-k relevant document chunks to form the context.
    2. Generate an answer using only the retrieved context (RAG).
    3. Evaluate the answer using LLM-as-a-judge on:
       - Groundedness: whether the answer is supported by the context.
       - Relevance: whether the answer addresses the question.

    Parameters:
        user_input (str): Input question.
        k (int): Number of retrieved chunks used as context.
        max_tokens (int): Maximum tokens for LLM responses.
        temperature (float): Controls randomness (0 = deterministic).
        top_p (float): Nucleus sampling threshold.
        top_k (int): Limits token sampling space.

    Returns:
        tuple:
            - groundedness (str): Score + explanation.
            - relevance (str): Score + explanation.
    """

    # Step 1 — Retrieve top-k relevant context chunks
    docs = retriever.invoke(user_input)
    context = "\n\n".join([d.page_content for d in docs[:k]])

    # Step 2 — Format prompt and generate RAG answer
    user_message = qna_user_message_template.format(
        context=context,
        question=user_input
    )

    rag_prompt = f"[INST] {qna_system_message}\n\n{user_message} [/INST]"

    response = llm(
        prompt=rag_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        stop=["</s>"]
    )

    # Extract generated answer
    answer = response["choices"][0]["text"].strip()

    # Step 3 — Evaluate groundedness (checks hallucination vs context)
    groundedness_prompt = f"""[INST] {groundedness_rater_system_message}

{user_message_template.format(context=context, question=user_input, answer=answer)} [/INST]"""

    grounded_response = llm(
        prompt=groundedness_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        stop=["</s>"]
    )

    # Step 4 — Evaluate relevance (checks alignment with question)
    relevance_prompt = f"""[INST] {relevance_rater_system_message}

{user_message_template.format(context=context, question=user_input, answer=answer)} [/INST]"""

    relevance_response = llm(
        prompt=relevance_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        stop=["</s>"]
    )

    # Return evaluation outputs (score + explanation)
    return (
        grounded_response["choices"][0]["text"].strip(),
        relevance_response["choices"][0]["text"].strip()
    )

### Query Evaluation

In [74]:
# Iterates through each question, generates RAG answers, and evaluates groundedness and relevance, displaying results.

# Store the results from the evaluation loop
results = []

for i, q in enumerate(questions, 1):
    grounded, relevance = generate_ground_relevance_response(q)

    result = {
        "Question": q,
        "Groundedness": grounded,
        "Relevance": relevance
    }

    results.append(result)

    print(f"--- Question {i} ---")
    print(f"Q: {q}")
    print(f"Groundedness {grounded}\n")
    print(f"Relevance {relevance}")
    print("--------------------------------------------------------------------------------\n")



--- Question 1 ---
Q: What is the protocol for managing sepsis in a critical care unit?
Groundedness Score: 3

Explanation: The answer is fully grounded in the context as it directly repeats the information provided in the context regarding the management of sepsis in a critical care unit, including obtaining specimens for culture and Gram stain before administering antibiotics, monitoring various physiologic parameters, administering corticosteroids to patients with septic shock, and using 0.9% saline for fluid resuscitation.

Relevance Score: 3

Explanation: The answer directly addresses the question by outlining the key steps in managing sepsis in a critical care unit, including obtaining specimens for culture and Gram stain, providing prompt empiric antibiotic therapy, monitoring various physiologic parameters, administering corticosteroids to patients with septic shock, and using 0.9% saline for fluid resuscitation. The response is on-topic and relevant to the context provided.
--

**Evaluation Summary:**

* **Groundedness**: All answers are **fully grounded** (score 3) as they are entirely supported by the provided context.
* **Relevance**: All answers are **fully relevant** (score 3) and directly address the questions.

The generated responses exhibit strong performance in both groundedness and relevance, with all answers fully aligned with the provided context and directly addressing the questions.


---
## <span style="color:#87CEEB;">**Actionable Insights and Business Recommendations**</span>

Below insights and recommendations align with the primary **objective of improving access to grounded and relevant medical knowledge** using the **RAG pipeline** for healthcare applications. The findings below provide key takeaways for ensuring the pipeline supports **clinicians, decision-makers, and patients** with accurate, timely, and accessible medical information.

1. **RAG Performance and Retrieval**

   * **Insight**: The use of **k=3** in the retrieval process ensures the most **relevant and contextual** medical information is retrieved, balancing between coverage and specificity. With **max_tokens=200**, the system generates answers that are **detailed yet concise**, reducing the risk of truncation and ensuring that the answers are comprehensive.
   * **Recommendation**: **Keep k=3** as it provides a **sufficient context** for clinical decision-making, while adjusting **top-k** to tighten the focus on key documents. This supports the objective of delivering **actionable and precise medical responses** without overwhelming users with excessive detail.

2. **Groundedness and Relevance Evaluation**

   * **Insight**: **Groundedness** and **relevance** scores consistently rated **fully grounded (3)** and **fully relevant (3)**, which is crucial for maintaining trust in the system’s accuracy. This ensures that the **model consistently produces clinically accurate responses** based on the **provided medical context**.
   * **Recommendation**: **Maintain current parameter settings** (k=3, max_tokens=200, temperature=0) as they consistently yield **high-quality outputs** aligned with project goals of improving **healthcare decision support**. Consider further **fine-tuning the model** with **additional datasets** to further enhance its reliability for specific medical domains.

3. **Model Prompt Effectiveness**

   * **Insight**: The **Clinical Guideline Mode** and **Explain Like I’m 15** prompts effectively balance **medical accuracy** and **clear communication**, ensuring that both healthcare professionals and patients can understand the content.
   * **Recommendation**: Continue using these **concise, clear instructions** for generating contextually accurate responses tailored to the user's needs. **Adapt prompt settings** for users based on their level of expertise to further **improve accessibility** of complex medical knowledge.

4. **Response Quality and Token Management**

   * **Insight**: The system's output remains **focused and relevant**, with no truncation issues, thanks to **max_tokens=200**. This ensures that responses are **comprehensive** without unnecessary verbosity.
   * **Recommendation**: **Monitor response length** and **adjust max_tokens** as needed to provide complete, actionable responses, especially for more complex medical queries. This supports the project objective of **efficiently delivering complex medical knowledge** without sacrificing quality or clarity.

5. **Scalability and Long-Term Impact**

   * **Insight**: The RAG pipeline is **scalable** and can adapt to increasing amounts of **clinical data** over time. It generates reliable answers across diverse clinical domains, ensuring long-term utility for healthcare professionals and patients.
   * **Recommendation**: Expand the **knowledge base** by continually integrating **up-to-date clinical guidelines** and **patient records** to ensure the system remains relevant and supports decision-making. This will align with the objective of providing **real-time, actionable insights** for clinical workflows.

### Conclusion

The **RAG pipeline** has proven its ability to meet the project objective of improving access to **grounded, relevant medical knowledge**. By fine-tuning parameters like **k=3**, **max_tokens**, and **top-k**, and by adapting the system prompts, we can continue enhancing the system's performance. With these improvements, the pipeline will be able to support **healthcare decision-making**, ensuring **accurate, timely, and accessible medical responses** that contribute to **improved patient outcomes** and **efficiency in clinical practices**.

---

In [78]:
import json
from copy import deepcopy

src = "Full_Code_NLP_RAG_Project_Notebook.ipynb"
dst = "Full_Code_NLP_RAG_Project_Notebook_clean.ipynb"

with open(src, "r", encoding="utf-8") as f:
    nb = json.load(f)

nb2 = deepcopy(nb)

if "metadata" in nb2 and "widgets" in nb2["metadata"]:
    del nb2["metadata"]["widgets"]

for cell in nb2.get("cells", []):
    if "outputs" in cell:
        for output in cell["outputs"]:
            if "data" in output:
                output["data"].pop("application/vnd.jupyter.widget-view+json", None)
                output["data"].pop("application/vnd.jupyter.widget-state+json", None)

with open(dst, "w", encoding="utf-8") as f:
    json.dump(nb2, f, ensure_ascii=False, indent=1)

print(f"Clean notebook written to {dst}")

Clean notebook written to Full_Code_NLP_RAG_Project_Notebook_clean.ipynb


In [76]:
import json

notebook_path = "Full_Code_NLP_RAG_Project_Notebook.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

# Remove top-level widget metadata
if "metadata" in nb and "widgets" in nb["metadata"]:
    del nb["metadata"]["widgets"]

# Remove widget outputs from cells
for cell in nb.get("cells", []):
    if "outputs" in cell:
        cleaned_outputs = []
        for output in cell["outputs"]:
            if "data" in output:
                output["data"].pop("application/vnd.jupyter.widget-view+json", None)
                output["data"].pop("application/vnd.jupyter.widget-state+json", None)
            cleaned_outputs.append(output)
        cell["outputs"] = cleaned_outputs

with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print("Widget metadata removed successfully.")

Widget metadata removed successfully.
